# Course Baseline: PCA + Linear SVM
**Owner: Felix / Zhangnan Fan.** Compare a course-method baseline with the team's CNNs on the unchanged patient split.

中文路线：图片 -> 像素向量 -> 仅在训练集拟合 PCA -> 线性 SVM -> 验证集选设置 -> 导出结果。

Course basis: Week 4 PCA (centering, principal directions and projection); Week 6 practical SVC(kernel='linear', C=...) and confusion matrices. The sklearn PCA implementation is a numerical implementation of the same projection; randomized SVD is explicitly configured for efficiency.

Plan before running: RGB 32x32, PCA dimensions {20,50}, SVM C {0.1,1.0}, fixed seed 42. Select by validation F1, then recall, then smaller dimension/C. These four candidates are a small declared comparison, not an exhaustive search. Test evaluation stays disabled until the team freezes its protocol.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Image as DisplayImage
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'data/splits/patient_split.csv').is_file(), 'Open this notebook from the repository or notebooks directory.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.baseline_utils import (load_manifest, choose_preprocessor, load_features,
    classification_metrics, prediction_table, export_figures, sha256_file)

SEED = 42
IMAGE_SIZE = 32
COMPONENTS = (20, 50)
C_VALUES = (0.1, 1.0)
SMOKE_RUN = False
RUN_TEST = False
TEAM_PROTOCOL_FROZEN = False
ALLOW_ANDERS_FALLBACK = True
local_file = ROOT / 'data/local_config.json'
local_config = json.loads(local_file.read_text()) if local_file.exists() else {}
DATA_ROOT = Path(os.environ.get('MALARIA_DATA_ROOT') or local_config.get('data_root', ROOT / 'data/cell_images'))
if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Set MALARIA_DATA_ROOT to the directory containing Parasitized and Uninfected, or set data/local_config.json.')
MANIFEST = ROOT / 'data/splits/patient_split.csv'
OUT = ROOT / 'artifacts/felix_svm' / ('smoke' if SMOKE_RUN else 'validation')
OUT.mkdir(parents=True, exist_ok=True)
_limits = threadpool_limits(limits=4)
print('Python:', sys.executable)
print('Output:', OUT)
print('Run type:', 'small execution check' if SMOKE_RUN else 'full training / validation comparison')
print('Test evaluation:', RUN_TEST)

## 1. Load the frozen manifest and check integrity
Do not generate a new split. Labels match the CNN notebooks: Uninfected=0, Parasitized=1.
患者分组、标签和文件路径先检查；数据审计可以查看测试集的文件清单，但此处不计算测试集模型表现。

In [ ]:
manifest, audit = load_manifest(MANIFEST, DATA_ROOT)
(OUT / 'split_audit.json').write_text(json.dumps(audit, indent=2))
display(manifest.groupby(['split','category']).size().unstack(fill_value=0))
print('Patient groups:', audit['patient_counts'])
print('Frozen CSV SHA256:', audit['manifest_sha256'])
content_audit = ROOT / 'artifacts/felix_svm/image_content_audit.json'
if content_audit.exists():
    content_report = json.loads(content_audit.read_text())
    print('Byte-content audit:', content_report)
    assert content_report['content_hashes_crossing_splits'] == 0
    assert content_report['conflicting_label_hashes'] == 0

train_rows = manifest[manifest.split == 'train'].reset_index(drop=True)
val_rows = manifest[manifest.split == 'val'].reset_index(drop=True)
if SMOKE_RUN:
    train_rows = train_rows.groupby('label',group_keys=False).sample(n=500,random_state=SEED).reset_index(drop=True)
    val_rows = val_rows.groupby('label',group_keys=False).sample(n=150,random_state=SEED).reset_index(drop=True)
print('Images used:', len(train_rows), 'train;', len(val_rows), 'validation')

## 2. Use the shared preprocessing interface
Prefer the team's src/preprocessing.py. At the fetched main commit this is still a stub, so the explicitly enabled fallback mirrors Anders's current RGB/bilinear/centred-black-padding/[0,1] implementation at 32x32.

Fallback use is reported in the results. It does not mean Shashank's implementation is finished. After that implementation is available, rerun and compare the transformed pixels before claiming equivalence. Scaling by the fixed constant 255 learns no validation statistics.

In [ ]:
first = train_rows.iloc[0]
load_image, preprocessing_backend = choose_preprocessor(
    DATA_ROOT / first.category / first.filename, IMAGE_SIZE,
    allow_legacy=ALLOW_ANDERS_FALLBACK)
print('Preprocessing backend:', preprocessing_backend)
example = load_image(DATA_ROOT / first.category / first.filename)
print('Image shape/range:', example.shape, float(example.min()), float(example.max()))

## 3. Turn images into vectors
32 x 32 RGB produces 3,072 pixel values per image. Every training image and every validation image is used in the full run; the smoke check is labelled separately.

In [ ]:
X_train = load_features(train_rows, DATA_ROOT, load_image)
X_val = load_features(val_rows, DATA_ROOT, load_image)
y_train = train_rows.label.to_numpy()
y_val = val_rows.label.to_numpy()
assert X_train.shape == (len(train_rows), IMAGE_SIZE * IMAGE_SIZE * 3)
assert X_val.shape == (len(val_rows), IMAGE_SIZE * IMAGE_SIZE * 3)
print('Feature shapes:', X_train.shape, X_val.shape)

## 4. Fit PCA on training images only
Week 4: centre the data, find principal directions, and project onto leading directions. PCA.fit learns the training mean and axes. Validation data only calls transform.

We fit the largest declared dimension once and compare prefixes of the same fitted basis, keeping the comparison reproducible. No whitening or additional learned pixel normalization is applied.

In [ ]:
started = time.perf_counter()
pca = PCA(n_components=max(COMPONENTS), svd_solver='randomized',
          iterated_power=4, random_state=SEED, whiten=False)
Z_train = pca.fit_transform(X_train)
Z_val = pca.transform(X_val)
np.testing.assert_allclose(pca.mean_, X_train.mean(axis=0), rtol=1e-5, atol=1e-6)
print('PCA fit seconds:', round(time.perf_counter()-started,2))
variance = np.cumsum(pca.explained_variance_ratio_)
for k in COMPONENTS: print(f'{k} components retain {variance[k-1]:.4f} of training variance')
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(np.arange(1,len(variance)+1),variance)
ax.set(xlabel='Number of principal components',ylabel='Cumulative training variance',title='PCA fitted on training images only')
fig.tight_layout(); fig.savefig(OUT/'pca_variance.png',dpi=180,bbox_inches='tight'); plt.show()

## 5. Train the declared linear SVM candidates
Week 6 practical uses sklearn.svm.SVC(kernel='linear', C=...). Fit uses training labels; validation metrics choose settings. Positive SVM decision scores favour Parasitized, but are not probabilities or confidence percentages.

In [ ]:
records, fitted_models = [], {}
for k in COMPONENTS:
    for c_value in C_VALUES:
        started = time.perf_counter()
        model = SVC(kernel='linear', C=c_value, cache_size=512, random_state=SEED)
        model.fit(Z_train[:, :k], y_train)
        pred = model.predict(Z_val[:, :k])
        scores = model.decision_function(Z_val[:, :k])
        row = {'n_components': k, 'C': c_value,
               **classification_metrics(y_val, pred, scores),
               'fit_and_validation_seconds': time.perf_counter()-started,
               'support_vectors': int(model.n_support_.sum())}
        records.append(row)
        fitted_models[(k,c_value)] = model
        pd.DataFrame(records).to_csv(OUT/'validation_candidates.csv',index=False)
        print(f'PCA={k}, C={c_value}: validation F1={row["f1"]:.4f}, accuracy={row["accuracy"]:.4f}',flush=True)
candidates = pd.DataFrame(records)
display(candidates.round(4))

## 6. Select and save one model using validation only
The deterministic tie-breaking rule was declared before fitting. This is a selected validation result; it is not an unbiased final test estimate.

In [ ]:
ranked = candidates.sort_values(['f1','recall','n_components','C'],
    ascending=[False,False,True,True],kind='mergesort')
best = ranked.iloc[0]
selected_k, selected_c = int(best.n_components), float(best.C)
selected_svm = fitted_models[(selected_k, selected_c)]
selection = {'method':'PCA + linear SVC','image_size':IMAGE_SIZE,
    'n_components':selected_k,'C':selected_c,'seed':SEED,
    'selection_rule':'validation F1 descending, recall descending, components ascending, C ascending',
    'manifest_sha256':audit['manifest_sha256'],
    'preprocessing_backend':preprocessing_backend,
    'shared_preprocessing_sha256':sha256_file(ROOT/'src/preprocessing.py'),
    'baseline_utils_sha256':sha256_file(ROOT/'src/baseline_utils.py'),
    'base_git_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),
    'smoke_run':SMOKE_RUN,'test_evaluated':False,
    'training_images':len(train_rows),'validation_images':len(val_rows),
    'versions':{p:package_metadata.version(p) for p in ['numpy','pandas','scikit-learn','Pillow','matplotlib']}}
(OUT/'selected_configuration.json').write_text(json.dumps(selection,indent=2))
model_path = OUT / 'pca_linear_svm.joblib'
joblib.dump({'pca':pca,'svm':selected_svm,'n_components':selected_k,
             'class_to_index':audit['class_to_index'],'configuration':selection},model_path)
print('Selected:',selected_k,'components; C =',selected_c)
print('Saved:',model_path)

## 7. Export validation predictions, metrics and reproducible figures
Every prediction retains filename, patient ID, true label, predicted label and decision score for group comparison. Cases are sampled by a fixed seed within TP/TN/FP/FN, rather than chosen for a preferred story.

In [ ]:
val_pred = selected_svm.predict(Z_val[:, :selected_k])
val_scores = selected_svm.decision_function(Z_val[:, :selected_k])
validation_metrics = classification_metrics(y_val,val_pred,val_scores)
validation_metrics.update({'split':'val','images':len(val_rows),
    'patient_groups':int(val_rows.patient_id.nunique()),
    'n_components':selected_k,'C':selected_c,
    'preprocessing_backend':preprocessing_backend,'test_evaluated':False})
(OUT/'validation_metrics.json').write_text(json.dumps(validation_metrics,indent=2))
val_table = prediction_table(val_rows,val_pred,val_scores)
val_table.to_csv(OUT/'validation_predictions.csv',index=False)
patient_errors = val_table.assign(error=~val_table.correct,
    fp=val_table.outcome.eq('FP'),fn=val_table.outcome.eq('FN')).groupby('patient_id').agg(
        images=('filename','size'),errors=('error','sum'),fp=('fp','sum'),fn=('fn','sum'))
patient_errors['error_rate'] = patient_errors.errors / patient_errors.images
patient_errors.to_csv(OUT/'patient_validation_errors.csv')
export_figures(val_table,DATA_ROOT,OUT,'Validation: PCA + linear SVM',seed=SEED)
display(pd.DataFrame([validation_metrics]).round(4))
display(DisplayImage(filename=str(OUT/'evaluation.png')))
display(DisplayImage(filename=str(OUT/'error_cases.png')))

## 8. Interpretation and limitations to discuss
- PCA maximizes retained image variance, not infection discrimination. Small 32x32 inputs may lose fine detail.
- Validation-selected scores may be optimistic. No test performance has been measured by the default notebook run.
- The fallback is a temporary integration choice, recorded explicitly. Shared-module pixel equivalence must be checked after integration.
- Compare only results using the same frozen manifest and label mapping. CNN inputs and training budgets differ and must be reported.
- SVM margins are not calibrated probabilities. Error images and Grad-CAM support exploratory analysis, not clinical safety or localization validation.
- The teacher requests learning curves where applicable; CNN epoch curves come from Anders. SVC has no epoch-based learning curve, so do not fabricate one.

## 9. Optional held-out test evaluation: disabled by default
Only enable both flags after the group agrees the preprocessing, model selection and analysis protocol are frozen. Do not use test scores to select parameters. The full default execution completes the validation deliverable while preserving the shared test holdout.

In [ ]:
if RUN_TEST:
    if SMOKE_RUN or not TEAM_PROTOCOL_FROZEN:
        raise RuntimeError('Test evaluation requires the full training run and an agreed frozen team protocol.')
    assert sha256_file(MANIFEST) == selection['manifest_sha256']
    test_rows = manifest[manifest.split=='test'].reset_index(drop=True)
    X_test = load_features(test_rows,DATA_ROOT,load_image)
    Z_test = pca.transform(X_test)[:, :selected_k]
    test_pred = selected_svm.predict(Z_test)
    test_scores = selected_svm.decision_function(Z_test)
    test_out = ROOT/'artifacts/felix_svm/test'
    test_out.mkdir(parents=True,exist_ok=True)
    test_table = prediction_table(test_rows,test_pred,test_scores)
    test_table.to_csv(test_out/'test_predictions.csv',index=False)
    test_metrics = classification_metrics(test_rows.label,test_pred,test_scores)
    (test_out/'test_metrics.json').write_text(json.dumps(test_metrics,indent=2))
    export_figures(test_table,DATA_ROOT,test_out,'Held-out test: PCA + linear SVM',seed=SEED)
    print(test_metrics)
else:
    print('Held-out test deliberately not evaluated. Coordinate the protocol freeze with the group.')

## 10. Learner checkpoint and team handoff
Before using these slides, explain in your own words: Why is PCA fitted on training images only? What does an FN mean when Parasitized=1? Why is the validation score not the final test score? Write one limitation and one question in your method card.

Deliver the validation predictions/configuration/figures to the group, coordinate the preprocessing update with Shashank, reuse Shania's CNN error plots with explicit model/split labels, and rehearse your own method/results section. Generated and executed files do not establish personal understanding or a completed presentation.